# ForestGuard

## 04 — Construcción del dataset para estimación de riesgo

En esta etapa se prepara un nuevo conjunto de datos orientado al problema principal de ForestGuard: estimar la probabilidad de que se registre un incendio forestal en un sector y periodo determinado.

Para ello se combinan los registros históricos de incendios con información meteorológica histórica.

La unidad de análisis utilizada será **cantón-día**.

La variable objetivo será:

- `1`: se registró al menos un incendio forestal.
- `0`: no se registró un incendio forestal en los datos disponibles para ese cantón y fecha.

## 1. Librerías

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import matplotlib.pyplot as plt

## 2. Carga del dataset de incendios

Se utiliza el dataset procesado en las etapas anteriores de Big Data.

Este archivo contiene los registros históricos de incendios forestales correspondientes a 2022–2025.

In [ ]:
from google.colab import files

archivo = "incendios_forestales_2022_2025_limpio.csv"

try:
    df = pd.read_csv(archivo)
except FileNotFoundError:
    print("Sube el archivo generado por el módulo 02:")
    uploaded = files.upload()
    nombre = list(uploaded.keys())[0]
    df = pd.read_csv(nombre)

print("Dimensiones:", df.shape)
display(df.head())

## 3. Preparación temporal

Las variables año, mes y día se combinan para crear una fecha completa.

La fecha permitirá relacionar los incendios históricos con las condiciones meteorológicas registradas en el mismo periodo.

In [ ]:
df["fecha"] = pd.to_datetime(
    {
        "year": df["año"],
        "month": df["mes"],
        "day": df["dia"]
    },
    errors="coerce"
)

print("Fechas inválidas:", df["fecha"].isna().sum())
print("Fecha inicial:", df["fecha"].min())
print("Fecha final:", df["fecha"].max())

## 4. Ubicación representativa por cantón

Para realizar las consultas meteorológicas se obtiene una coordenada representativa para cada cantón.

Como aproximación para este prototipo, se utiliza la mediana de las coordenadas disponibles en los registros históricos de incendios de cada cantón.

Esta aproximación puede mejorarse posteriormente utilizando centroides geográficos oficiales.

In [ ]:
df_ubicacion = (
    df
    .dropna(subset=["provincia", "canton", "latitud", "longitud"])
    .groupby(["provincia", "canton"], as_index=False)
    .agg(
        latitud=("latitud", "median"),
        longitud=("longitud", "median")
    )
)

print("Cantones disponibles:", len(df_ubicacion))
display(df_ubicacion.head())

## 5. Creación de la variable objetivo

La variable objetivo del modelo será `incendio`.

Cada observación representa un cantón en una fecha determinada:

- `incendio = 1`: existe al menos un incendio registrado en ese cantón y fecha.
- `incendio = 0`: no existe un incendio registrado en los datos disponibles para ese cantón y fecha.

Esta transformación permite trabajar posteriormente con un problema de clasificación binaria y obtener una estimación de probabilidad.

In [ ]:
eventos_dia = (
    df
    .dropna(subset=["fecha", "provincia", "canton"])
    .groupby(["provincia", "canton", "fecha"])
    .size()
    .reset_index(name="cantidad_incendios")
)

eventos_dia["incendio"] = 1

print("Observaciones cantón-día con incendio registrado:", len(eventos_dia))
display(eventos_dia.head())

## 6. Construcción de observaciones cantón-día

Se genera un calendario para los cantones disponibles durante el periodo analizado.

Esto permite incluir tanto fechas con incendios registrados como fechas sin incendios registrados.

**Importante:** `incendio = 0` significa que no existe un incendio registrado en la fuente disponible para ese cantón y fecha; no demuestra que nunca haya ocurrido un evento no reportado.

In [ ]:
fecha_inicio = df["fecha"].min()
fecha_fin = df["fecha"].max()

fechas = pd.DataFrame({
    "fecha": pd.date_range(
        start=fecha_inicio,
        end=fecha_fin,
        freq="D"
    )
})

ubicacion_tmp = df_ubicacion.copy()
ubicacion_tmp["key"] = 1
fechas["key"] = 1

calendario = (
    ubicacion_tmp
    .merge(fechas, on="key")
    .drop(columns="key")
)

calendario = calendario.merge(
    eventos_dia[
        ["provincia", "canton", "fecha", "incendio"]
    ],
    on=["provincia", "canton", "fecha"],
    how="left"
)

calendario["incendio"] = (
    calendario["incendio"]
    .fillna(0)
    .astype(int)
)

print("Observaciones generadas:", len(calendario))
display(calendario["incendio"].value_counts().to_frame("Cantidad"))

## 7. Obtención de información meteorológica histórica

Se consulta información meteorológica histórica para cada cantón utilizando sus coordenadas representativas.

Las variables iniciales seleccionadas son:

- temperatura media;
- humedad relativa media;
- precipitación acumulada;
- velocidad máxima del viento.

Estas variables representan condiciones ambientales que pueden analizarse en relación con la ocurrencia de incendios forestales.

In [ ]:
URL_CLIMA = "https://archive-api.open-meteo.com/v1/archive"

fecha_inicio_api = fecha_inicio.strftime("%Y-%m-%d")
fecha_fin_api = fecha_fin.strftime("%Y-%m-%d")

def obtener_clima(fila):
    params = {
        "latitude": float(fila["latitud"]),
        "longitude": float(fila["longitud"]),
        "start_date": fecha_inicio_api,
        "end_date": fecha_fin_api,
        "daily": (
            "temperature_2m_mean,"
            "relative_humidity_2m_mean,"
            "precipitation_sum,"
            "wind_speed_10m_max"
        ),
        "timezone": "America/Guayaquil"
    }

    respuesta = requests.get(
        URL_CLIMA,
        params=params,
        timeout=60
    )
    respuesta.raise_for_status()

    datos = respuesta.json()["daily"]
    clima = pd.DataFrame(datos)

    clima["provincia"] = fila["provincia"]
    clima["canton"] = fila["canton"]

    return clima

### Ejecución de las consultas meteorológicas

Se realiza una consulta por cada cantón y se integran los resultados en un único conjunto de datos meteorológicos.

Si una consulta falla, el proceso continúa con los demás cantones y muestra el error correspondiente.

In [ ]:
resultados_clima = []

for _, fila in df_ubicacion.iterrows():
    try:
        clima = obtener_clima(fila)
        resultados_clima.append(clima)
        print(f"OK: {fila['provincia']} - {fila['canton']}")
    except Exception as e:
        print(f"Error en {fila['canton']}: {e}")

    time.sleep(0.2)

if not resultados_clima:
    raise RuntimeError("No se pudo obtener información meteorológica.")

df_clima = pd.concat(
    resultados_clima,
    ignore_index=True
)

print("Registros meteorológicos:", len(df_clima))
display(df_clima.head())

## 8. Preparación de las variables meteorológicas

Se renombran las variables para facilitar su interpretación dentro del proyecto.

In [ ]:
df_clima = df_clima.rename(
    columns={
        "time": "fecha",
        "temperature_2m_mean": "temperatura_media",
        "relative_humidity_2m_mean": "humedad_media",
        "precipitation_sum": "precipitacion",
        "wind_speed_10m_max": "viento_max"
    }
)

df_clima["fecha"] = pd.to_datetime(df_clima["fecha"])

display(df_clima.head())

## 9. Integración del dataset de riesgo

El calendario de observaciones cantón-día se combina con la información meteorológica utilizando como claves la provincia, el cantón y la fecha.

El resultado constituye el dataset que será utilizado posteriormente para Machine Learning.

In [ ]:
df_riesgo = calendario.merge(
    df_clima,
    on=["provincia", "canton", "fecha"],
    how="left"
)

df_riesgo["año"] = df_riesgo["fecha"].dt.year
df_riesgo["mes"] = df_riesgo["fecha"].dt.month
df_riesgo["dia_semana"] = df_riesgo["fecha"].dt.dayofweek

print("Dimensiones:", df_riesgo.shape)
display(df_riesgo.head())

## 10. Validación del dataset para Machine Learning

Antes de guardar el dataset se revisa su estructura, valores faltantes, registros duplicados y distribución de la variable objetivo.

In [ ]:
print("Duplicados:")
print(df_riesgo.duplicated().sum())

print("\nValores faltantes:")
display(
    df_riesgo
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Faltantes")
)

print("\nVariable objetivo:")
display(
    df_riesgo["incendio"]
    .value_counts()
    .to_frame("Cantidad")
)

print("\nProporción:")
display(
    df_riesgo["incendio"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("Porcentaje")
)

## 11. Distribución de la variable objetivo

In [ ]:
df_riesgo["incendio"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(6, 4)
)

plt.title("Distribución de la variable objetivo")
plt.xlabel("Incendio registrado")
plt.ylabel("Cantidad de observaciones")
plt.xticks([0, 1], ["No", "Sí"], rotation=0)
plt.show()

## 12. Guardado del dataset para Machine Learning

El dataset final contiene observaciones de cantón-día, variables meteorológicas y la variable objetivo `incendio`.

Este archivo será utilizado como entrada para los modelos de Machine Learning.

In [ ]:
archivo_riesgo = "forestguard_dataset_riesgo_2022_2025.csv"

df_riesgo.to_csv(
    archivo_riesgo,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset guardado:", archivo_riesgo)
print("Dimensiones finales:", df_riesgo.shape)

In [ ]:
from google.colab import files
files.download("forestguard_dataset_riesgo_2022_2025.csv")

## 13. Conclusión

Se construyó un dataset orientado a la estimación de riesgo de incendio forestal mediante observaciones cantón-día.

El nuevo conjunto integra información histórica de incendios con variables meteorológicas y crea una variable objetivo binaria que diferencia entre fechas con y sin incendios registrados.

Este dataset permite continuar con una etapa de Machine Learning enfocada directamente en estimar la probabilidad de registro de incendios forestales.